# Practica final LLM

## 1. Carga de librerias

In [1]:
import os
import json
import time
import requests
import random
from typing import List, Dict, Any, Optional
from datetime import datetime
from dotenv import load_dotenv
import tiktoken
import numpy as np
from openai import OpenAI

# Para la parte de RAG
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document

# Para la carga del PDF
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

# Para logging
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

C:\Users\diego\AppData\Local\Temp\ipykernel_39684\3859828662.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import OpenAIEmbeddings


### 1.1 Conf API-Keys

In [ ]:
# Cargar desde .env (crea un archivo .env con OPENAI_API_KEY=tu_clave)
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    OPENAI_API_KEY = input("Introduce tu OpenAI API Key: ")
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Configuración del modelo (parámetros expuestos según criterios)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    OPENAI_API_KEY = input("Introduce tu OpenAI API Key: ")
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Parámetros del modelo (expuestos claramente)
MODEL_CONFIG = {
    "model": "gpt-4.1-nano",  # o "gpt-4.1-mini"
    "temperature": 0.7,
    "max_tokens": 500,
    "top_p": 0.9
}

client = OpenAI()

: 

In [3]:
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
models = client.models.list()

for model in models:
    print(model.id)

2026-06-16 18:26:39,885 - INFO - HTTP Request: GET https://api.openai.com/v1/models "HTTP/1.1 200 OK"


text-embedding-3-small
gpt-4.1-mini
gpt-4.1-nano


### 1.2 Carga del PDF

In [4]:
def cargar_guia_desde_pdf(ruta_pdf: str) -> str:
    """
    Carga el contenido completo de un PDF.
    Args:
        ruta_pdf: Ruta al archivo PDF
    Returns:
        str: Texto completo extraído del PDF
    """
    if not Path(ruta_pdf).exists():
        raise FileNotFoundError(f"No se encuentra el PDF: {ruta_pdf}")
    
    loader = PyPDFLoader(ruta_pdf)
    documentos = loader.load()
    
    texto_completo = "\n\n".join([doc.page_content for doc in documentos])
    logger.info(f"PDF cargado: {ruta_pdf} - {len(documentos)} páginas, {len(texto_completo)} caracteres")
    
    print(f"Guía cargada: {ruta_pdf}")
    
    return texto_completo

NOMBRE_PDF = "TENERIFE.pdf" 

try:
    GUIA_TURISTICA_TEXTO = cargar_guia_desde_pdf(NOMBRE_PDF)
except FileNotFoundError as e:
    print(f"{e}")
    print("El PDF debe estar en la misma carpeta que este notebook.")

2026-06-16 18:26:52,543 - INFO - PDF cargado: TENERIFE.pdf - 25 páginas, 16142 caracteres


Guía cargada: TENERIFE.pdf


## 2. Conf CHUNKING, EMBEDDINGS Y VECTOR STORE

In [10]:
def crear_chunks(texto: str, chunk_size: int = 900, overlap: int = 120) -> List[Document]:
    """
    Divide el texto en chunks usando RecursiveCharacterTextSplitter. 
    Args:
        texto: Texto completo a dividir
        chunk_size: Tamaño máximo de cada chunk
        overlap: Superposición entre chunks   
    Returns:
        List[Document]: Lista de documentos chunkenizados
    """
    logger.info(f"Creando chunks: tamaño={chunk_size}, overlap={overlap}")
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    
    documento = Document(page_content=texto, metadata={"source": NOMBRE_PDF})
    chunks = splitter.split_documents([documento])
    
    # Añadir metadatos
    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i
        chunk.metadata["source_name"] = NOMBRE_PDF
    
    logger.info(f"Chunks creados: {len(chunks)}")
    return chunks


# ============================================
# 2.2 EMBEDDINGS Y VECTOR STORE
# ============================================

def crear_vector_store(chunks: List[Document]) -> InMemoryVectorStore:
    """
    Crea un vector store InMemoryVectorStore a partir de los chunks.
    Args:
        chunks: Lista de Documentos
    Returns:
        InMemoryVectorStore: Vector store en memoria
    """
    logger.info("Generando embeddings y creando vector store")
    
    # Embeddings
    EMBEDDING_MODEL = "text-embedding-3-small"
    
    embeddings = OpenAIEmbeddings(
        model=EMBEDDING_MODEL,
        openai_api_key=OPENAI_API_KEY
    )
    
    # InMemoryVectorStore
    vectorstore = InMemoryVectorStore(embeddings)
    
    # Añadir documentos
    vectorstore.add_documents(chunks)
    
    logger.info(f"Vector store creado con {len(chunks)} documentos")
    
    return vectorstore


# ============================================
# 2.3 FUNCIÓN DE RECUPERACIÓN
# ============================================

def recuperar_contexto(pregunta: str, vectorstore: InMemoryVectorStore, k: int = 3) -> str:
    """
    Recupera los fragmentos más relevantes para una pregunta.    
    Args:
        pregunta: Texto de la consulta
        vectorstore: Vector store InMemoryVectorStore
        k: Número de fragmentos a recuperar  
    Returns:
        str: Contexto formateado con fuentes
    """
    logger.info(f"Recuperando contexto para: {pregunta[:50]}...")
    
    # Búsqueda por similitud
    docs_recuperados = vectorstore.similarity_search(query=pregunta, k=k)
    
    contexto = ""
    for i, doc in enumerate(docs_recuperados, 1):
        fuente = doc.metadata.get("source_name", NOMBRE_PDF)
        chunk_id = doc.metadata.get("chunk_id", "?")
        contexto += f"\n[Fuente {i}: {fuente}, chunk {chunk_id}]\n{doc.page_content}\n"
    
    logger.info(f"Recuperados {len(docs_recuperados)} fragmentos")
    
    return contexto


# ============================================
# 2.4 EJECUCIÓN DE LA INDEXACIÓN
# ============================================

print("\n" + "="*60)
print(" INDEXACIÓN RAG")
print("="*60)

# Crear chunks
chunks = crear_chunks(GUIA_TURISTICA_TEXTO)
print(f"{len(chunks)} chunks creados")

# Crear vector store (InMemoryVectorStore como en clase)
vectorstore = crear_vector_store(chunks)
print(f"Vector store InMemoryVectorStore creado con {len(chunks)} documentos")

# Prueba de recuperación
print("\nPrueba de recuperación:")
pregunta_test = "¿Donde puedo ir a comer?"
contexto_test = recuperar_contexto(pregunta_test, vectorstore, k=2)
print("\nRespuesta:")
print(contexto_test)

2026-06-16 18:30:59,830 - INFO - Creando chunks: tamaño=900, overlap=120
2026-06-16 18:30:59,834 - INFO - Chunks creados: 27
2026-06-16 18:30:59,838 - INFO - Generando embeddings y creando vector store



 INDEXACIÓN RAG
27 chunks creados


2026-06-16 18:31:00,404 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-16 18:31:00,435 - INFO - Vector store creado con 27 documentos
2026-06-16 18:31:00,435 - INFO - Recuperando contexto para: ¿Donde puedo ir a comer?...


Vector store InMemoryVectorStore creado con 27 documentos

Prueba de recuperación:


2026-06-16 18:31:00,712 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-16 18:31:00,722 - INFO - Recuperados 2 fragmentos



Respuesta:

[Fuente 1: TENERIFE.pdf, chunk 25]
Guachinche es un sitio como el que he indicado.  
 
▪ Pizzería Gioconda (zona de La Orotava [ubicación]) [Tripadvisor] 
▪ El Palestra (zona de La Orotava [ubicación]. Este es el típico sitio que 
ha estado toda la vida en La Orotava y que te pides a domicilio cuando 
tienes el antojo. Nosotros solo pedimos una cosa: Plato feliz de pizza + 
Nestea de mango-piña para beber. [Tripadvisor] 
▪ Sitios de bocadillos top: Cafetería Deyfe - Bar-Cafetería Australia - Bar 
El Paraíso (espectacular la salsa de aguacate de aquí. 
Recomendación: menú feliz de bocadillo de pollo + Nestea de 
mango-piña) - Cafetería Zumería Mango - El Track 
 
 
Nota:   los bocadillos de pollo del norte de Tenerife no son como los bocadillos 
que se hacen en otros lugares de España.   Recomiendo que lo prueben, está 
increíble.

[Fuente 2: TENERIFE.pdf, chunk 2]
o Callejear dirección Plaza del Adelantado [ubicación] y tomar algo por ahí si 
hace buen tiempo (si queréis c

## 3 Memoria conversacional

In [11]:
class MemoriaConversacional:
    """
    Gestiona el historial de conversación con control de tokens. 
    Attributes:
        historial: Lista de mensajes (rol + contenido)
        max_tokens: Límite máximo de tokens en memoria
        tokenizer: Tokenizador para contar tokens
    """
    
    def __init__(self, max_tokens_contexto: int = 3000):
        """
        Inicializa la memoria conversacional. 
        Args:
            max_tokens_contexto: Límite máximo de tokens
        """
        self.historial: List[Dict[str, str]] = []
        self.max_tokens = max_tokens_contexto
        self.tokenizer = tiktoken.encoding_for_model(MODEL_CONFIG["model"])
        logger.info(f"Memoria inicializada: max_tokens={max_tokens_contexto}")
    
    def agregar_interaccion(self, pregunta: str, respuesta: str) -> None:
        """
        Añade una interacción al historial
        Args:
            pregunta: Texto del usuario
            respuesta: Texto del asistente
        """
        self.historial.append({"role": "user", "content": pregunta})
        self.historial.append({"role": "assistant", "content": respuesta})
        self._controlar_longitud()
        logger.info(f"Memoria actualizada: {len(self.historial)} mensajes")
    
    def _controlar_longitud(self) -> None:
        """Recorta el historial si supera el límite de tokens."""
        tokens_totales = sum(
            len(self.tokenizer.encode(m["content"])) 
            for m in self.historial
        )
        
        while tokens_totales > self.max_tokens and len(self.historial) > 4:
            # Eliminar el par más antiguo (user + assistant)
            self.historial.pop(0)
            self.historial.pop(0)
            tokens_totales = sum(
                len(self.tokenizer.encode(m["content"])) 
                for m in self.historial
            )
        
        if tokens_totales > self.max_tokens:
            logger.warning(f"Memoria excede límite: {tokens_totales} > {self.max_tokens}")
    
    def obtener_historial(self, ultimos_n: Optional[int] = None) -> List[Dict[str, str]]:
        """
        Devuelve el historial conversacional.
        Args:
            ultimos_n: Si se especifica, devuelve solo los últimos N mensajes
        Returns:
            List[Dict]: Lista de mensajes
        """
        if ultimos_n:
            return self.historial[-ultimos_n:]
        return self.historial
    
    def limpiar(self) -> None:
        """Reinicia la conversación."""
        self.historial = []
        logger.info("Memoria limpiada")
    
    def __len__(self) -> int:
        return len(self.historial)

## 4 Función para respuestas RAG y Memoria

In [12]:
def responder_con_rag_y_memoria(
    pregunta: str, 
    memoria: MemoriaConversacional,
    vectorstore: vectorstore 
) -> str:
    """
    Genera una respuesta usando RAG y memoria conversacional. 
    Args:
        pregunta: Texto del usuario
        memoria: Objeto MemoriaConversacional
        vectorstore: Vector store InMemoryVectorStore
    Returns:
        str: Respuesta del asistente
    """
    logger.info(f"Respondiendo a: {pregunta[:50]}...")
    
    # 1. Recuperar contexto
    contexto = recuperar_contexto(pregunta, vectorstore)
    
    # 2. Construir mensajes
    messages = [
        {
            "role": "system", 
            "content": (
                "Eres un asistente turístico experto en Tenerife. "
                "Responde usando la información del contexto proporcionado. "
                "Si no encuentras la respuesta en el contexto, dilo claramente. "
                "Cita siempre las fuentes entre corchetes [Fuente X]. "
                "Sé amable, útil y da recomendaciones prácticas."
                "Realiza siempre una sugerencia a continuación de cada respuesta"
            )
        }
    ]
    
    # 3. Añadir historial (últimos 5 mensajes para contexto)
    messages.extend(memoria.obtener_historial(ultimos_n=5))
    
    # 4. Añadir pregunta actual con contexto
    messages.append({
        "role": "user",
        "content": f"Contexto de la guía:\n{contexto}\n\nPregunta del usuario: {pregunta}"
    })
    
    # 5. Llamada al modelo
    response = client.chat.completions.create(
        model=MODEL_CONFIG["model"],
        messages=messages,
        temperature=MODEL_CONFIG["temperature"],
        max_tokens=MODEL_CONFIG["max_tokens"],
        top_p=MODEL_CONFIG["top_p"]
    )
    
    respuesta = response.choices[0].message.content
    
    # 6. Guardar en memoria
    memoria.agregar_interaccion(pregunta, respuesta)
    
    logger.info(f"Respuesta generada: {len(respuesta)} caracteres")
    
    return respuesta

## 5 Function Calling. (Revisa permisos acceso al Teide).

In [13]:
# Esquema JSON de la herramienta (obligatorio según criterios)
TEIDE_PERMIT_TOOL = {
    "type": "function",
    "function": {
        "name": "get_teide_access_status",
        "description": (
            "Consulta la disponibilidad de permisos para subir al Teide. "
            "El Teide requiere autorización del Cabildo de Tenerife con cupo limitado. "
            "Útil cuando el usuario pregunta sobre subir al Teide, permisos, "
            "reservas o cómo llegar a la cima."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "fecha": {
                    "type": "string",
                    "description": "Fecha deseada en formato YYYY-MM-DD. Ejemplo: 2025-07-15"
                },
                "num_personas": {
                    "type": "integer",
                    "description": "Número de personas (máximo 15 por grupo y minimo de 1)",
                    "minimum": 1,
                    "maximum": 15,
                    "default": 2
                }
            },
            "required": ["fecha"]
        }
    }
}


# ============================================
# 5.2 IMPLEMENTACIÓN DE LA FUNCIÓN
# ============================================

def get_teide_access_status(fecha: str, num_personas: int = 2) -> str:
    """
    Simula la disponibilidad de permisos para subir al Teide.
    Basado en reglas reales del Parque Nacional del Teide:
    - Límite de 200 personas/día
    - Reducción en fines de semana y verano
    - Recomendación de reserva anticipada   
    Args:
        fecha: Fecha en formato YYYY-MM-DD
        num_personas: Número de personas (máx 15)
    Returns:
        str: JSON con disponibilidad y recomendaciones
    """
    logger.info(f"Consultando permisos Teide: fecha={fecha}, personas={num_personas}")
    
    # Validar formato de fecha
    try:
        fecha_dt = datetime.strptime(fecha, "%Y-%m-%d")
    except ValueError:
        return json.dumps({
            "error": "Formato de fecha inválido. Usa YYYY-MM-DD"
        })
    
    # Reglas reales del Teide
    mes = fecha_dt.month
    dia_semana = fecha_dt.weekday()
    es_finde = dia_semana >= 5  # sábado=5, domingo=6
    es_verano = mes in [6, 7, 8, 9]
    es_invierno = mes in [12, 1, 2, 3]
    
    # Capacidad diaria real: 200 personas/día
    capacidad_total = 200
    
    # Reducciones reales
    if es_verano and es_finde:
        capacidad_total = 100  # Verano + fin de semana = alta demanda
    elif es_verano or es_finde:
        capacidad_total = 150
    elif es_invierno and not es_finde:
        capacidad_total = 250  # Mas capacidad en invierno laborable
    
    # Simular ocupación realista
    if es_verano:
        ocupacion_porcentaje = random.randint(75, 98)
    elif es_finde:
        ocupacion_porcentaje = random.randint(70, 95)
    else:
        ocupacion_porcentaje = random.randint(30, 80)
    
    disponibles = max(0, capacidad_total - int(capacidad_total * ocupacion_porcentaje / 100))
    
    # Generar respuesta con mensaje adecuado
    if num_personas <= disponibles:
        estado = "disponible"
        mensaje = (
            f"¡Felicidades! Hay {disponibles} permisos disponibles "
            f"para el {fecha}. Puedes solicitar {num_personas} plaza(s)."
        )
        recomendacion = "Reserva cuanto antes en la web oficial del Cabildo."
    elif disponibles > 0:
        estado = "limitado"
        mensaje = (
            f"Quedan solo {disponibles} permisos para el {fecha}, "
            f"pero necesitas {num_personas}. Prueba otra fecha o grupos más pequeños."
        )
        recomendacion = "Considera un día laborable o temporada baja (octubre-noviembre)."
    else:
        estado = "agotado"
        mensaje = (
            f"No quedan permisos para el {fecha}. "
            f"El Teide tiene límite de {capacidad_total} personas/día."
        )
        recomendacion = "Prueba con 2-3 meses de antelación o evita fines de semana y verano."
    
    # Información adicional
    horarios = {
        "invierno": "De 9:00 a 16:00 (último acceso 13:00)",
        "verano": "De 9:00 a 17:00 (último acceso 14:00)"
    }
    horario_texto = horarios["verano"] if es_verano else horarios["invierno"]
    
    resultado = {
        "fecha": fecha,
        "permisos_disponibles": disponibles,
        "permisos_solicitados": num_personas,
        "estado": estado,
        "mensaje": mensaje,
        "recomendacion": recomendacion,
        "horario_subida": horario_texto,
        "url_oficial": "https://www.reservasparquenacionaldelteide.es/",
        "consejo_adicional": (
            "Lleva calzado de montaña y ropa de abrigo. "
            "El teleférico requiere entrada aparte (38€ ida y vuelta)."
        )
    }
    
    logger.info(f"Permisos Teide: {estado} - {disponibles} disponibles")
    return json.dumps(resultado, ensure_ascii=False)

In [14]:
def responder_con_tools(
    pregunta: str, 
    memoria: MemoriaConversacional,
    vectorstore: vectorstore 
) -> str:
    """
    Responde pudiendo llamar a la función get_teide_access_status.
    Registra todos los intentos (requisito de logging).
    Args:
        pregunta: Texto del usuario
        memoria: Objeto MemoriaConversacional
        vectorstore: Vector store InMemoryVectorStore 
    Returns:
        str: Respuesta final del asistente
    """
    logger.info(f"Respondiendo con tools: {pregunta[:50]}...")
    
    # 1. Recuperar contexto RAG
    contexto = recuperar_contexto(pregunta, vectorstore)
    
    # 2. Construir mensajes
    messages = [
        {
            "role": "system",
            "content": (
                "Eres un asistente turístico experto en Tenerife. "
                "Puedes usar la función get_teide_access_status para consultar "
                "disponibilidad de permisos para subir al Teide. "
                "Cuando respondas, se util y cita fuentes de la guía cuando sea relevante."
                "Cuando respondas, en caso de estar agotado, sugiere planes alternativos."
            )
        }
    ]
    
    messages.extend(memoria.obtener_historial(ultimos_n=6))
    messages.append({
        "role": "user",
        "content": f"Contexto de guía:\n{contexto}\n\nPregunta del usuario: {pregunta}"
    })
    
    # 3. Primera llamada al modelo (puede solicitar tool)
    response = client.chat.completions.create(
        model=MODEL_CONFIG["model"],
        messages=messages,
        tools=[TEIDE_PERMIT_TOOL],
        tool_choice="auto",
        temperature=MODEL_CONFIG["temperature"]
    )
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls
    
    # 4. Procesar tool calls si existen
    if tool_calls:
        for tool_call in tool_calls:
            if tool_call.function.name == "get_teide_access_status":
                args = json.loads(tool_call.function.arguments)
                fecha = args.get("fecha")
                num_personas = args.get("num_personas", 2)
                
                logger.info(
                    f"Tool call ejecutado: get_teide_access_status("
                    f"fecha='{fecha}', personas={num_personas})"
                )
                
                # Ejecutar la función
                resultado_tool = get_teide_access_status(fecha, num_personas)
                
                # Segunda llamada con el resultado
                messages.append(response_message)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": resultado_tool
                })
                
                # Generar respuesta final
                final_response = client.chat.completions.create(
                    model=MODEL_CONFIG["model"],
                    messages=messages,
                    temperature=MODEL_CONFIG["temperature"],
                    max_tokens=MODEL_CONFIG["max_tokens"]
                )
                
                respuesta = final_response.choices[0].message.content
                memoria.agregar_interaccion(pregunta, respuesta)
                return respuesta
    
    # 5. Sin tool call
    respuesta = response_message.content
    memoria.agregar_interaccion(pregunta, respuesta)
    return respuesta

## 6 Evaluación

In [15]:
def evaluar_requisitos(vectorstore: vectorstore ) -> Dict[str, Any]:
    """
    Evalúa los tres componentes obligatorios:
    1. RAG: recuperación de información del PDF
    2. Memoria multiturno: coherencia entre turnos
    Args:
        vectorstore: Vector store InMemoryVectorStore
    Returns:
        Dict: Resultados de la evaluación
    """
    print("\n" + "="*60)
    print("EVALUACIÓN AUTOMÁTICA DE REQUISITOS")
    print("="*60)
    
    memoria = MemoriaConversacional()
    resultados = {
        "RAG": False,
        "Memoria_multiturno": False,
        "Function_call": False,
        "tool_calls_exitosas": 0,
        "detalles": {}
    }
    
    # ---------- TEST 1: RAG ----------
    print("\nTEST 1: RAG (recuperación de información)")
    
    preguntas_rag = [
        "¿Cuánto cuesta la entrada al Loro Parque?",
        "¿Cómo se llega al Teide?",
        "¿Qué playas hay en Tenerife?"
    ]
    
    aciertos_rag = 0
    for pregunta in preguntas_rag:
        contexto = recuperar_contexto(pregunta, vectorstore, k=2)
        # Verificar que recuperó algo relevante
        if len(contexto.strip()) > 20:
            aciertos_rag += 1
    
    if aciertos_rag >= 2:
        resultados["RAG"] = True
        print(f"RAG correcto ({aciertos_rag}/3 preguntas recuperadas)")
    else:
        print(f"RAG insuficiente ({aciertos_rag}/3)")
    
    resultados["detalles"]["RAG"] = f"{aciertos_rag}/3 aciertos"
    
    # ---------- TEST 2: MEMORIA MULTITURNO ----------
    print("\nTEST 2: Memoria multiturno")
    
    memoria.limpiar()
    
    # Primera pregunta
    r1 = responder_con_rag_y_memoria(
        "¿Qué recomiendas comer en Tenerife?", 
        memoria, 
        vectorstore
    )
    
    # Segunda pregunta (depende del contexto de la primera)
    r2 = responder_con_rag_y_memoria(
        "¿Y dónde puedo probar ese plato típico?",
        memoria,
        vectorstore
    )
    
    # Verificar coherencia
    palabras_clave = ["restaurante", "dónde", "comer", "probar", "típico", "recomiendo"]
    if any(palabra in r2.lower() for palabra in palabras_clave):
        resultados["Memoria_multiturno"] = True
        print("Memoria funciona: respuesta coherente")

In [16]:
evaluar_requisitos(vectorstore)


EVALUACIÓN AUTOMÁTICA DE REQUISITOS


2026-06-16 18:32:18,564 - INFO - Memoria inicializada: max_tokens=3000
2026-06-16 18:32:18,564 - INFO - Recuperando contexto para: ¿Cuánto cuesta la entrada al Loro Parque?...



TEST 1: RAG (recuperación de información)


2026-06-16 18:32:18,941 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-16 18:32:18,955 - INFO - Recuperados 2 fragmentos
2026-06-16 18:32:18,957 - INFO - Recuperando contexto para: ¿Cómo se llega al Teide?...
2026-06-16 18:32:19,150 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-16 18:32:19,155 - INFO - Recuperados 2 fragmentos
2026-06-16 18:32:19,155 - INFO - Recuperando contexto para: ¿Qué playas hay en Tenerife?...
2026-06-16 18:32:19,355 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-16 18:32:19,359 - INFO - Recuperados 2 fragmentos
2026-06-16 18:32:19,359 - INFO - Memoria limpiada
2026-06-16 18:32:19,371 - INFO - Respondiendo a: ¿Qué recomiendas comer en Tenerife?...
2026-06-16 18:32:19,371 - INFO - Recuperando contexto para: ¿Qué recomiendas comer en Tenerife?...


RAG correcto (3/3 preguntas recuperadas)

TEST 2: Memoria multiturno


2026-06-16 18:32:19,753 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-16 18:32:19,771 - INFO - Recuperados 3 fragmentos
2026-06-16 18:32:22,423 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-16 18:32:22,454 - INFO - Memoria actualizada: 2 mensajes
2026-06-16 18:32:22,454 - INFO - Respuesta generada: 1378 caracteres
2026-06-16 18:32:22,454 - INFO - Respondiendo a: ¿Y dónde puedo probar ese plato típico?...
2026-06-16 18:32:22,454 - INFO - Recuperando contexto para: ¿Y dónde puedo probar ese plato típico?...
2026-06-16 18:32:22,727 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-16 18:32:22,745 - INFO - Recuperados 3 fragmentos
2026-06-16 18:32:29,281 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-16 18:32:29,281 - INFO - Memoria actualizada: 4 mensajes
2026-06-16 18:32:29,296 - INFO - Respuesta generad

Memoria funciona: respuesta coherente


## 7 GUI para chat

In [72]:
def chat_turistico():
    """Bucle conversacional para probar el asistente."""
    memoria = MemoriaConversacional()
    print("===Asistente Turístico ===\nEscribe 'salir' para terminar.\n")
    
    while True:
        user_input = input("Tú: ")
        if user_input.lower() in ["salir", "exit", "quit"]:
            print("¡Hasta luego!")
            break
        
        # Detectar si quiere clima (para forzar tool call)
        if user_input.lower().startswith("clima:"):
            ciudad = user_input.split(":", 1)[1].strip()
            pregunta = f"¿Qué tiempo hace en {ciudad}?"
        else:
            pregunta = user_input
        
        print("Asistente: ", end="")
        respuesta = responder_con_tools(pregunta, memoria, vectorstore)
        print(respuesta)
        print("Asistente: ", end="")
        
        # Opcional: streaming
        # responder_streaming(pregunta, memoria)

# Descomentar para usar
chat_turistico()

2026-06-16 16:22:47,010 - INFO - Memoria inicializada: max_tokens=3000


===Asistente Turístico ===
Escribe 'salir' para terminar.



Tú:  Consulta disponibilidad para subir al Teide el día 2026-07-04 para 3 personas


2026-06-16 16:22:49,011 - INFO - Respondiendo con tools: Consulta disponibilidad para subir al Teide el día...
2026-06-16 16:22:49,011 - INFO - Recuperando contexto para: Consulta disponibilidad para subir al Teide el día...


Asistente: 

2026-06-16 16:22:49,323 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-16 16:22:49,332 - INFO - Recuperados 3 fragmentos
2026-06-16 16:22:51,366 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-16 16:22:51,374 - INFO - Tool call ejecutado: get_teide_access_status(fecha='2026-07-04', personas=3)
2026-06-16 16:22:51,374 - INFO - Consultando permisos Teide: fecha=2026-07-04, personas=3
2026-06-16 16:22:51,376 - INFO - Permisos Teide: disponible - 4 disponibles
2026-06-16 16:22:54,431 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-16 16:22:54,448 - INFO - Memoria actualizada: 2 mensajes


¡Buenas noticias! Para el 4 de julio de 2026, hay 4 permisos disponibles para subir al Teide, y puedes solicitar 3 plazas. Te recomiendo reservar cuanto antes en la web oficial del Cabildo: [reservasparquenacionaldelteide.es](https://www.reservasparquenacionaldelteide.es/), ya que suele llenarse rápidamente. 

Recuerda que la subida en teleférico cuesta 38€ ida y vuelta y que el horario de acceso es de 9:00 a 17:00, con el último acceso a las 14:00. Además, es recomendable llevar calzado de montaña y ropa de abrigo, especialmente si quieres disfrutar de la vista desde arriba o hacer una subida nocturna para observar las estrellas, ¡una experiencia espectacular!

Si por alguna razón no pudieras conseguir los permisos, te sugiero visitar el Centro de Visitantes de El Portillo para aprender más sobre el volcán y el parque. También puedes explorar otras áreas de la isla como las playas de Costa Adeje o el Mirador de La Tarta, que ofrecen vistas impresionantes y experiencias únicas.
Asisten

Tú:  salir


¡Hasta luego!
